In [59]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import re

In [156]:
df = pd.read_csv('D:/Study/data_science/underpriced-listing-predictor/data/02.parsed/all_appliances_parsed.csv')
print("Shape:", df.shape)
print("Columns:", df.columns.tolist())
print("\nFeatures list length distribution per category:")
df['n_features'] = df['features'].str.count("',") + 1
print(df.groupby('category')['n_features'].describe())
print("\n--- Sample rows ---")
for cat in df['category'].unique():
    print(f"\n=== {cat} ===")
    print(df[df['category']==cat].iloc[0][['name', 'price', 'rating', 'features']].to_dict())

Shape: (2925, 5)
Columns: ['name', 'price', 'rating', 'category', 'features']

Features list length distribution per category:
                 count       mean       std  min   25%   50%   75%   max
category                                                                
AC              1020.0  10.945098  1.139412  5.0  10.0  11.0  12.0  15.0
Refrigerator    1020.0   8.255882  1.433585  2.0   8.0   8.0   9.0  11.0
WashingMachine   885.0  11.027119  1.348432  4.0  10.0  11.0  12.0  18.0

--- Sample rows ---

=== AC ===
{'name': 'Whirlpool SAI18B52MCD1 1.5 Ton 5 Star Inverter Split AC', 'price': '₹24,990', 'rating': '--rating: 4.65;', 'features': "['Split AC', '1.5 Ton Capacity', 'Air Swing', 'Auto Restart', 'Turbo Mode', 'Sleep Mode', 'Dust Filter', 'Self Diagnosis', 'Night Glow Buttons', '5 Star Rating']"}

=== Refrigerator ===
{'name': 'Samsung RT28C3733S8 236 L 3 Star Double Door Refrigerator', 'price': '₹25,490', 'rating': '--rating:4.3;', 'features': "['Multi Door', 'Top Mounted F

In [157]:
# Let us check for null values
print(df.isnull().sum()) # Zero null values in our data

name          0
price         0
rating        0
category      0
features      0
n_features    0
dtype: int64


- No Null values found

### ***1. Cleaning Price Column***

In [158]:
df['price'] = pd.to_numeric(df['price'].str.replace('[₹,]', '', regex=True), errors='coerce')

print(df.info())
print(df['price'].sample(5))

# Check how many rows became NaN during coercion
nans = df['price'].isna().sum()
print(f"Price cleaning: {nans} rows coerced to NaN")

# Check for any remaining non-numeric values (sanity check)
if nans > 0:
    print("Warning: Some prices were not converted. Inspecting first few NaNs:")
    print(df[df['price'].isna()].head())

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   name        2925 non-null   str  
 1   price       2925 non-null   int64
 2   rating      2925 non-null   str  
 3   category    2925 non-null   str  
 4   features    2925 non-null   str  
 5   n_features  2925 non-null   int64
dtypes: int64(2), str(4)
memory usage: 882.1 KB
None
614     32999
940     35900
1459    16990
1792    68890
103     27990
Name: price, dtype: int64
Price cleaning: 0 rows coerced to NaN


### ***2. Cleaning Rating Column***

In [159]:
df['rating'] = pd.to_numeric(df['rating'].str.split(':').str[1].str.strip().str.replace(';','' , regex=True),errors='coerce')
print(df.info())
print(df['rating'].sample(5))

# Check how many rows became NaN during coercion
nans = df['rating'].isna().sum()
print(f"Rating cleaning: {nans} rows coerced to NaN")

# Check for any remaining non-numeric values (sanity check)
if nans > 0:
    print("Warning: Some ratings were not converted. Inspecting first few NaNs:")
    print(df[df['rating'].isna()].head())

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   name        2925 non-null   str    
 1   price       2925 non-null   int64  
 2   rating      2925 non-null   float64
 3   category    2925 non-null   str    
 4   features    2925 non-null   str    
 5   n_features  2925 non-null   int64  
dtypes: float64(1), int64(2), str(3)
memory usage: 841.0 KB
None
1293    4.00
2654    4.25
626     4.15
869     4.40
2732    4.35
Name: rating, dtype: float64
Rating cleaning: 0 rows coerced to NaN


### ***3. Cleaning Name Column***

**(i) Brand Name***

In [160]:
# 1. Extract brand name (first word)
df['brand_name'] = df['name'].str.split(" ").str[0]

# 2. Define the correction map
brand_corrections = {
    "O": "O General",
    "Blue": "Blue Star"
    # Only if they are different
    # Add the other broken brands you found here...
}

# 3. Apply the correction
df['brand_name'] = df['brand_name'].replace(brand_corrections)

# 4. Verify the results
print("Brand name distribution:")
print(df['brand_name'].value_counts().head(10))

Brand name distribution:
brand_name
Haier        315
Samsung      269
LG           264
Whirlpool    256
Godrej       234
Voltas       219
Blue Star    166
IFB          151
Lloyd        142
Panasonic    123
Name: count, dtype: int64


**(ii) Capacity Column**

In [166]:
# This assigns the two captured groups directly to new columns in your df
df[['capacity_value', 'capacity_unit']] = df['features'].str.extract(r'(\d+\.?\d*)\s*(Ton|-Ton|L|-L|Kg|-Kg)',re.IGNORECASE)

# Now convert the value column to float, because extract returns everything as strings
df['capacity_value'] = pd.to_numeric(df['capacity_value'], errors='coerce')

In [184]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 9 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   name            2925 non-null   str    
 1   price           2925 non-null   int64  
 2   rating          2925 non-null   float64
 3   category        2925 non-null   str    
 4   features        2925 non-null   str    
 5   n_features      2925 non-null   int64  
 6   brand_name      2925 non-null   object 
 7   capacity_value  2925 non-null   float64
 8   capacity_unit   2925 non-null   str    
dtypes: float64(2), int64(2), object(1), str(4)
memory usage: 915.3+ KB


In [179]:
idx =df[(df['capacity_value'].isna())].index.to_list()

In [181]:
df.loc[idx, ['capacity_value', 'capacity_unit']] = [[2.02, 'Ton'],[30,'kg']]
df[(df['capacity_value'].isna())]


,name,price,rating,category,features,n_features,brand_name,capacity_value,capacity_unit,Price,Rating


In [199]:
df['capacity_unit'].value_counts()

capacity_unit
Ton    1020
L      1020
kg      883
Kg        2
Name: count, dtype: int64

In [206]:
idx = df[df['capacity_unit']=='Kg'].index.to_list()
df.loc[idx,'capacity_unit']='kg'

In [207]:
df['capacity_unit'].value_counts()

capacity_unit
Ton    1020
L      1020
kg      885
Name: count, dtype: int64

***(ii) Extract Star Rating from Name***

In [188]:
# Extract star rating from name using regex - handles:
# - Digits followed by optional spaces, optional hyphen, optional spaces, then "star" (case-insensitive)
# Examples: "5 Star", "4-Star", "3-star", "2 star", etc.
df['star_rating_from_name'] = df['name'].str.extract(r'(\d+)\s*-?\s*star', expand=False, flags=re.IGNORECASE)

# Convert to numeric (handle any non-numeric values)
df['star_rating_from_name'] = pd.to_numeric(df['star_rating_from_name'], errors='coerce')


**Finding Star Ratings From Features**

In [189]:
df['star_rating_from_features'] = df['features'].str.extract(r'(\d+)\s*-?\s*star', expand=False, flags=re.IGNORECASE)
df['star_rating_from_features'] = pd.to_numeric(df['star_rating_from_features'], errors='coerce')

In [190]:
# 3. Fill NaNs in the main column using the features-extracted values
df['star_rating'] = df['star_rating_from_name'].fillna(df['star_rating_from_features'])

In [195]:
# Add binary indicator for categories that typically have energy star ratings
df['has_star_rating'] = df['category'].isin(['AC', 'Refrigerator']).astype(int)


In [213]:
idx = df[df['category']=='WashingMachine'].index.to_list()
df.loc[idx,'star_rating'] = np.nan
# filled all the star ratings of WashingMachine as nan 

In [ ]:
# drop the temporary columns
del df['star_rating_from_features'] ,df['star_rating_from_name']

In [216]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             2925 non-null   str    
 1   price            2925 non-null   int64  
 2   rating           2925 non-null   float64
 3   category         2925 non-null   str    
 4   features         2925 non-null   str    
 5   n_features       2925 non-null   int64  
 6   brand_name       2925 non-null   object 
 7   capacity_value   2925 non-null   float64
 8   capacity_unit    2925 non-null   str    
 9   star_rating      1957 non-null   float64
 10  has_star_rating  2925 non-null   int64  
dtypes: float64(3), int64(3), object(1), str(4)
memory usage: 961.0+ KB


**(iii) **

In [219]:
df['name']

0       Whirlpool SAI18B52MCD1 1.5 Ton 5 Star Inverter...
1       Carrier CAI18ER3R34F0 1.5 Ton 3 Star 2024 Inve...
2       O General ASGG18CGAB-B 1.5 Ton 5 Star Inverter...
3       Haier HSU18V-POW5BN-INV 1.5 Ton 5 Star 2025 In...
4       Voltas 183V XAZX 1.5 Ton 3 Star Split Inverter AC
                              ...                        
2920    Whirlpool Ace Super Soak 7.5 kg Semi Automatic...
2921    Whirlpool ACE 7.5 Turbo Dry 7.5kg Semi Automat...
2922    IFB TLSDG Aqua 7Kg Fully Automatic Top Load Wa...
2923    Whirlpool Bloom Wash 360  World Series 80H 8kg...
2924    Godrej WT Eon 700 PF 7Kg Top Loading Washing M...
Name: name, Length: 2925, dtype: str